# ✈️ ANÁLISIS AVANZADO DE VUELOS CON DATOS REALES

## 🎯 **OBJETIVO**
Análisis completo de datos de vuelos usando **2.7 millones de registros reales** para demostrar todos los métodos avanzados de DataFrames en Spark.

## 📊 **DATASETS**
- **✈️ Vuelos**: 2.7M+ registros con retrasos, aerolíneas, rutas
- **🛫 Aeropuertos**: 367 aeropuertos con ubicaciones geográficas

## 🔍 **ANÁLISIS QUE REALIZAREMOS**
1. **📈 Análisis de Retrasos** - Agregaciones complejas
2. **🗺️ Rutas Más Populares** - Joins y groupBy
3. **🏆 Aerolíneas con Mejor Puntualidad** - Window Functions
4. **📅 Análisis Temporal** - Funciones de fecha avanzadas
5. **🌍 Análisis Geográfico** - Filtros por estado/ciudad
6. **⚡ Optimización de Consultas** - Rendimiento con datos grandes

---

## 🔧 **CONFIGURACIÓN INICIAL**


In [ ]:
# 🔄 CELDA DE REINICIO - Ejecutar si hay errores de SparkContext
try:
    if 'spark' in globals():
        print("🔄 Cerrando sesión anterior de Spark...")
        spark.stop()
        print("✅ Sesión anterior cerrada")
except:
    print("ℹ️ No había sesión anterior")

if 'spark' in globals():
    del spark

print("🚀 Listo para crear nueva sesión de Spark optimizada para datos grandes")


In [ ]:
# Importar todas las librerías necesarias para análisis de datos grandes
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import socket
import os

print("📚 Librerías importadas correctamente para análisis avanzado")


In [ ]:
# Crear SparkSession optimizada para datos grandes (2.7M registros)
def get_spark_master():
    try:
        hostname = socket.gethostname()
        if 'jupyter' in hostname or 'master' in hostname or 'jupyterlab' in hostname:
            return "spark://master:7077"
        else:
            return "spark://localhost:7077"
    except:
        return "local[*]"

spark_master_url = get_spark_master()
print(f"🔧 Conectando a: {spark_master_url}")

spark = SparkSession.builder \
    .appName("Analisis-Vuelos-Avanzado") \
    .master(spark_master_url) \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.instances", "2") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

print("✅ SparkSession creada y optimizada para análisis de datos grandes")
print(f"📊 Memoria del driver: {spark.conf.get('spark.driver.memory')}")
print(f"📊 Memoria del executor: {spark.conf.get('spark.executor.memory')}")
print(f"📊 Cores del executor: {spark.conf.get('spark.executor.cores')}")


In [ ]:
# 📊 CARGA DE DATOS REALES
print("=" * 60)
print("📊 CARGA DE DATOS REALES DE VUELOS")
print("=" * 60)

# 1. CARGAR DATOS DE AEROPUERTOS
print("🛫 Cargando datos de aeropuertos...")
airports_path = "/user/data/etapa3/airports.csv"

# Definir esquema para aeropuertos
airports_schema = StructType([
    StructField("airport_id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("name", StringType(), True)
])

df_airports = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .schema(airports_schema) \
    .csv(airports_path)

print(f"✅ Aeropuertos cargados: {df_airports.count()} registros")

# 2. CARGAR DATOS DE VUELOS
print("✈️ Cargando datos de vuelos...")
flights_path = "/user/data/etapa3/raw-flight-data.csv"

# Definir esquema para vuelos
flights_schema = StructType([
    StructField("DayofMonth", IntegerType(), True),
    StructField("DayOfWeek", IntegerType(), True),
    StructField("Carrier", StringType(), True),
    StructField("OriginAirportID", IntegerType(), True),
    StructField("DestAirportID", IntegerType(), True),
    StructField("DepDelay", IntegerType(), True),
    StructField("ArrDelay", IntegerType(), True)
])

df_flights = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .schema(flights_schema) \
    .csv(flights_path)

print(f"✅ Vuelos cargados: {df_flights.count():,} registros")

# 3. MOSTRAR MUESTRAS DE DATOS
print("\n📋 Muestra de datos de aeropuertos:")
df_airports.show(10, truncate=False)

print("\n📋 Muestra de datos de vuelos:")
df_flights.show(10, truncate=False)

# 4. INFORMACIÓN DE ESQUEMAS
print("\n📋 Esquema de aeropuertos:")
df_airports.printSchema()

print("\n📋 Esquema de vuelos:")
df_flights.printSchema()


## 📈 **ANÁLISIS 1: ANÁLISIS DE RETRASOS DE VUELOS**

### **🔍 Objetivo:**
Analizar los retrasos de vuelos usando agregaciones complejas y estadísticas avanzadas.


In [ ]:
# 📈 ANÁLISIS DE RETRASOS DE VUELOS
print("=" * 60)
print("📈 ANÁLISIS DE RETRASOS DE VUELOS")
print("=" * 60)

# 1. ESTADÍSTICAS GENERALES DE RETRASOS
print("📊 Estadísticas generales de retrasos:")
df_delay_stats = df_flights.select(
    count("DepDelay").alias("total_vuelos"),
    avg("DepDelay").alias("promedio_retraso_salida"),
    avg("ArrDelay").alias("promedio_retraso_llegada"),
    max("DepDelay").alias("max_retraso_salida"),
    max("ArrDelay").alias("max_retraso_llegada"),
    min("DepDelay").alias("min_retraso_salida"),
    min("ArrDelay").alias("min_retraso_llegada"),
    stddev("DepDelay").alias("desviacion_retraso_salida"),
    stddev("ArrDelay").alias("desviacion_retraso_llegada")
)

df_delay_stats.show(truncate=False)

# 2. ANÁLISIS DE RETRASOS POR AEROLÍNEA
print("\n📊 Retrasos por aerolínea:")
df_airline_delays = df_flights.groupBy("Carrier").agg(
    count("Carrier").alias("total_vuelos"),
    avg("DepDelay").alias("promedio_retraso_salida"),
    avg("ArrDelay").alias("promedio_retraso_llegada"),
    max("DepDelay").alias("max_retraso_salida"),
    stddev("DepDelay").alias("desviacion_retraso_salida")
).orderBy(col("promedio_retraso_salida").desc())

df_airline_delays.show(truncate=False)

# 3. ANÁLISIS DE RETRASOS POR DÍA DE LA SEMANA
print("\n📊 Retrasos por día de la semana:")
df_day_delays = df_flights.groupBy("DayOfWeek").agg(
    count("DayOfWeek").alias("total_vuelos"),
    avg("DepDelay").alias("promedio_retraso_salida"),
    avg("ArrDelay").alias("promedio_retraso_llegada"),
    max("DepDelay").alias("max_retraso_salida")
).orderBy("DayOfWeek")

# Agregar nombres de días
df_day_names = df_day_delays.withColumn("dia_nombre",
    when(col("DayOfWeek") == 1, "Lunes")
    .when(col("DayOfWeek") == 2, "Martes")
    .when(col("DayOfWeek") == 3, "Miércoles")
    .when(col("DayOfWeek") == 4, "Jueves")
    .when(col("DayOfWeek") == 5, "Viernes")
    .when(col("DayOfWeek") == 6, "Sábado")
    .when(col("DayOfWeek") == 7, "Domingo")
)

df_day_names.select("DayOfWeek", "dia_nombre", "total_vuelos", 
                   "promedio_retraso_salida", "promedio_retraso_llegada").show(truncate=False)

# 4. VUELOS CON RETRASOS EXTREMOS
print("\n📊 Vuelos con retrasos extremos (>120 minutos):")
df_extreme_delays = df_flights.filter(
    (col("DepDelay") > 120) | (col("ArrDelay") > 120)
).select("Carrier", "OriginAirportID", "DestAirportID", 
         "DepDelay", "ArrDelay", "DayOfWeek")

print(f"📊 Total de vuelos con retrasos extremos: {df_extreme_delays.count():,}")

df_extreme_delays.orderBy(col("DepDelay").desc()).show(10, truncate=False)

# 5. PUNTUALIDAD GENERAL
print("\n📊 Análisis de puntualidad:")
df_puntualidad = df_flights.withColumn("puntual_salida", col("DepDelay") <= 0) \
                          .withColumn("puntual_llegada", col("ArrDelay") <= 0)

puntualidad_stats = df_puntualidad.agg(
    count("*").alias("total_vuelos"),
    sum(when(col("puntual_salida"), 1).otherwise(0)).alias("vuelos_puntuales_salida"),
    sum(when(col("puntual_llegada"), 1).otherwise(0)).alias("vuelos_puntuales_llegada")
)

puntualidad_stats = puntualidad_stats.withColumn(
    "porcentaje_puntual_salida", 
    (col("vuelos_puntuales_salida") / col("total_vuelos") * 100).cast("decimal(5,2)")
).withColumn(
    "porcentaje_puntual_llegada", 
    (col("vuelos_puntuales_llegada") / col("total_vuelos") * 100).cast("decimal(5,2)")
)

puntualidad_stats.show(truncate=False)


## 🗺️ **ANÁLISIS 2: RUTAS MÁS POPULARES**

### **🔍 Objetivo:**
Identificar las rutas más populares y analizar el tráfico aéreo usando joins y agregaciones.


In [ ]:
# 🗺️ ANÁLISIS DE RUTAS MÁS POPULARES
print("=" * 60)
print("🗺️ ANÁLISIS DE RUTAS MÁS POPULARES")
print("=" * 60)

# 1. RUTAS MÁS FRECUENTADAS (sin información de aeropuertos)
print("📊 Top 10 rutas más frecuentadas:")
df_routes = df_flights.groupBy("OriginAirportID", "DestAirportID").agg(
    count("*").alias("total_vuelos"),
    avg("DepDelay").alias("promedio_retraso_salida"),
    avg("ArrDelay").alias("promedio_retraso_llegada")
).orderBy(col("total_vuelos").desc())

df_routes.show(10, truncate=False)

# 2. RUTAS MÁS POPULARES CON INFORMACIÓN DE AEROPUERTOS
print("\n📊 Top 10 rutas con información de aeropuertos:")
df_routes_with_airports = df_flights \
    .join(df_airports.alias("origin"), df_flights.OriginAirportID == df_airports.airport_id) \
    .join(df_airports.alias("dest"), df_flights.DestAirportID == df_airports.airport_id) \
    .groupBy(
        col("origin.city").alias("ciudad_origen"),
        col("origin.state").alias("estado_origen"),
        col("dest.city").alias("ciudad_destino"),
        col("dest.state").alias("estado_destino")
    ).agg(
        count("*").alias("total_vuelos"),
        avg("DepDelay").alias("promedio_retraso_salida"),
        avg("ArrDelay").alias("promedio_retraso_llegada")
    ).orderBy(col("total_vuelos").desc())

df_routes_with_airports.show(10, truncate=False)

# 3. RUTAS MÁS POPULARES POR ESTADO
print("\n📊 Top 10 rutas interestatales más populares:")
df_interstate_routes = df_flights \
    .join(df_airports.alias("origin"), df_flights.OriginAirportID == df_airports.airport_id) \
    .join(df_airports.alias("dest"), df_flights.DestAirportID == df_airports.airport_id) \
    .filter(col("origin.state") != col("dest.state")) \
    .groupBy(
        col("origin.state").alias("estado_origen"),
        col("dest.state").alias("estado_destino")
    ).agg(
        count("*").alias("total_vuelos"),
        avg("DepDelay").alias("promedio_retraso_salida")
    ).orderBy(col("total_vuelos").desc())

df_interstate_routes.show(10, truncate=False)

# 4. AEROPUERTOS CON MÁS TRÁFICO
print("\n📊 Top 10 aeropuertos con más tráfico (origen + destino):")

# Tráfico como origen
df_origin_traffic = df_flights.groupBy("OriginAirportID").agg(
    count("*").alias("vuelos_origen")
)

# Tráfico como destino
df_dest_traffic = df_flights.groupBy("DestAirportID").agg(
    count("*").alias("vuelos_destino")
)

# Combinar y calcular tráfico total
df_total_traffic = df_origin_traffic \
    .join(df_dest_traffic, df_origin_traffic.OriginAirportID == df_dest_traffic.DestAirportID) \
    .withColumn("total_trafico", col("vuelos_origen") + col("vuelos_destino")) \
    .join(df_airports, col("OriginAirportID") == df_airports.airport_id) \
    .select(
        col("airport_id"),
        col("name"),
        col("city"),
        col("state"),
        col("vuelos_origen"),
        col("vuelos_destino"),
        col("total_trafico")
    ).orderBy(col("total_trafico").desc())

df_total_traffic.show(10, truncate=False)

# 5. ANÁLISIS DE CONECTIVIDAD
print("\n📊 Análisis de conectividad - Aeropuertos con más destinos únicos:")
df_connectivity = df_flights.groupBy("OriginAirportID").agg(
    countDistinct("DestAirportID").alias("destinos_unicos"),
    count("*").alias("total_vuelos")
).join(df_airports, col("OriginAirportID") == df_airports.airport_id) \
 .select("airport_id", "name", "city", "state", "destinos_unicos", "total_vuelos") \
 .orderBy(col("destinos_unicos").desc())

df_connectivity.show(10, truncate=False)


## 🏆 **ANÁLISIS 3: RENDIMIENTO DE AEROLÍNEAS**

### **🔍 Objetivo:**
Analizar el rendimiento de las aerolíneas usando Window Functions y rankings.


In [ ]:
# 🏆 ANÁLISIS DE RENDIMIENTO DE AEROLÍNEAS
print("=" * 60)
print("🏆 ANÁLISIS DE RENDIMIENTO DE AEROLÍNEAS")
print("=" * 60)

# 1. RANKING DE AEROLÍNEAS POR PUNTUALIDAD
print("📊 Ranking de aerolíneas por puntualidad:")
df_airline_performance = df_flights.groupBy("Carrier").agg(
    count("*").alias("total_vuelos"),
    avg("DepDelay").alias("promedio_retraso_salida"),
    avg("ArrDelay").alias("promedio_retraso_llegada"),
    sum(when(col("DepDelay") <= 0, 1).otherwise(0)).alias("vuelos_puntuales_salida"),
    sum(when(col("ArrDelay") <= 0, 1).otherwise(0)).alias("vuelos_puntuales_llegada")
).withColumn(
    "porcentaje_puntualidad_salida", 
    (col("vuelos_puntuales_salida") / col("total_vuelos") * 100).cast("decimal(5,2)")
).withColumn(
    "porcentaje_puntualidad_llegada", 
    (col("vuelos_puntuales_llegada") / col("total_vuelos") * 100).cast("decimal(5,2)")
)

# Usar Window Functions para ranking
window_puntualidad = Window.orderBy(col("porcentaje_puntualidad_llegada").desc())
df_airline_ranking = df_airline_performance.withColumn(
    "rank_puntualidad", rank().over(window_puntualidad)
).orderBy(col("porcentaje_puntualidad_llegada").desc())

df_airline_ranking.show(truncate=False)

# 2. AEROLÍNEAS CON MEJOR Y PEOR RENDIMIENTO
print("\n📊 Top 5 aerolíneas con mejor puntualidad:")
df_airline_ranking.filter(col("total_vuelos") > 1000).show(5, truncate=False)

print("\n📊 Top 5 aerolíneas con peor puntualidad:")
df_airline_ranking.filter(col("total_vuelos") > 1000).orderBy(col("porcentaje_puntualidad_llegada").asc()).show(5, truncate=False)

# 3. ANÁLISIS DE CONSISTENCIA (desviación estándar)
print("\n📊 Análisis de consistencia de aerolíneas:")
df_consistency = df_flights.groupBy("Carrier").agg(
    count("*").alias("total_vuelos"),
    avg("DepDelay").alias("promedio_retraso_salida"),
    stddev("DepDelay").alias("desviacion_retraso_salida"),
    avg("ArrDelay").alias("promedio_retraso_llegada"),
    stddev("ArrDelay").alias("desviacion_retraso_llegada")
).filter(col("total_vuelos") > 1000) \
 .withColumn("coeficiente_variacion_salida", 
            (col("desviacion_retraso_salida") / abs(col("promedio_retraso_salida"))).cast("decimal(5,2)")) \
 .withColumn("coeficiente_variacion_llegada", 
            (col("desviacion_retraso_llegada") / abs(col("promedio_retraso_llegada"))).cast("decimal(5,2)")) \
 .orderBy(col("desviacion_retraso_llegada").asc())

df_consistency.show(truncate=False)

# 4. COMPARACIÓN CON PROMEDIO GENERAL
print("\n📊 Comparación con promedio general:")
promedio_general = df_flights.agg(
    avg("DepDelay").alias("promedio_general_salida"),
    avg("ArrDelay").alias("promedio_general_llegada")
).collect()[0]

print(f"📊 Promedio general de retraso de salida: {promedio_general['promedio_general_salida']:.2f} minutos")
print(f"📊 Promedio general de retraso de llegada: {promedio_general['promedio_general_llegada']:.2f} minutos")

# Aerolíneas por encima y por debajo del promedio
df_vs_promedio = df_airline_performance.withColumn(
    "vs_promedio_salida", 
    (col("promedio_retraso_salida") - promedio_general['promedio_general_salida']).cast("decimal(5,2)")
).withColumn(
    "vs_promedio_llegada", 
    (col("promedio_retraso_llegada") - promedio_general['promedio_general_llegada']).cast("decimal(5,2)")
).filter(col("total_vuelos") > 1000) \
 .orderBy(col("vs_promedio_llegada").asc())

print("\n📊 Aerolíneas vs promedio general (mejor a peor):")
df_vs_promedio.select("Carrier", "total_vuelos", "promedio_retraso_llegada", "vs_promedio_llegada").show(truncate=False)


## 📊 **ANÁLISIS 4: OPTIMIZACIÓN Y RENDIMIENTO**

### **🔍 Objetivo:**
Demostrar técnicas de optimización y análisis de rendimiento con datos grandes.


In [ ]:
# 📊 ANÁLISIS DE OPTIMIZACIÓN Y RENDIMIENTO
print("=" * 60)
print("📊 ANÁLISIS DE OPTIMIZACIÓN Y RENDIMIENTO")
print("=" * 60)

# 1. CACHE DE DATAFRAMES FRECUENTEMENTE USADOS
print("🔄 Cacheando DataFrames para optimización...")
df_flights_cached = df_flights.cache()
df_airports_cached = df_airports.cache()

# Forzar evaluación del cache
df_flights_cached.count()
df_airports_cached.count()

print("✅ DataFrames cachead en memoria")

# 2. ANÁLISIS DE PARTICIONES
print(f"\n📊 Número de particiones del DataFrame de vuelos: {df_flights_cached.rdd.getNumPartitions()}")
print(f"📊 Número de particiones del DataFrame de aeropuertos: {df_airports_cached.rdd.getNumPartitions()}")

# 3. CONSULTA COMPLEJA PARA ANÁLISIS DE RENDIMIENTO
print("\n🔍 Ejecutando consulta compleja para análisis de rendimiento...")

# Consulta compleja con múltiples joins y agregaciones
df_complex_query = df_flights_cached \
    .join(df_airports_cached.alias("origin"), df_flights_cached.OriginAirportID == df_airports_cached.airport_id) \
    .join(df_airports_cached.alias("dest"), df_flights_cached.DestAirportID == df_airports_cached.airport_id) \
    .filter(col("DepDelay") > 0) \
    .filter(col("ArrDelay") > 0) \
    .groupBy(
        col("Carrier"),
        col("origin.state").alias("estado_origen"),
        col("dest.state").alias("estado_destino")
    ).agg(
        count("*").alias("vuelos_con_retraso"),
        avg("DepDelay").alias("promedio_retraso_salida"),
        avg("ArrDelay").alias("promedio_retraso_llegada"),
        max("DepDelay").alias("max_retraso_salida"),
        max("ArrDelay").alias("max_retraso_llegada")
    ).filter(col("vuelos_con_retraso") > 100) \
    .orderBy(col("promedio_retraso_llegada").desc())

print("📋 Resultado de consulta compleja (top 10):")
df_complex_query.show(10, truncate=False)

# 4. PLAN DE EJECUCIÓN
print("\n📊 Plan de ejecución de la consulta compleja:")
df_complex_query.explain()

# 5. ESTADÍSTICAS DE RENDIMIENTO
print(f"\n📊 Estadísticas de rendimiento:")
print(f"   • Total de vuelos procesados: {df_flights_cached.count():,}")
print(f"   • Total de aeropuertos: {df_airports_cached.count()}")
print(f"   • Vuelos con retrasos: {df_flights_cached.filter(col('DepDelay') > 0).count():,}")

# 6. ANÁLISIS DE MEMORIA Y CACHE
print(f"\n📊 Estado del caché:")
print(f"   • Vuelos en caché: {df_flights_cached.storageLevel}")
print(f"   • Aeropuertos en caché: {df_airports_cached.storageLevel}")

# 7. MEJORES PRÁCTICAS APLICADAS
print("\n✅ Mejores prácticas aplicadas:")
print("   • Cache de DataFrames reutilizados")
print("   • Filtrado temprano de datos")
print("   • Uso de alias para joins múltiples")
print("   • Agregaciones eficientes")
print("   • Configuración optimizada de Spark")

# 8. LIMPIAR CACHE
df_flights_cached.unpersist()
df_airports_cached.unpersist()
print("🗑️ Caché liberado para liberar memoria")


In [ ]:
# 🔒 Cerrar SparkSession
spark.stop()
print("🔒 SparkSession cerrada correctamente")
print("🎉 ¡Análisis completo de vuelos finalizado!")
print("\n📊 RESUMEN DE ANÁLISIS REALIZADOS:")
print("   ✅ Análisis de retrasos de vuelos")
print("   ✅ Rutas más populares")
print("   ✅ Rendimiento de aerolíneas")
print("   ✅ Optimización y rendimiento")
print("   ✅ Procesamiento de 2.7M+ registros")
print("\n🚀 ¡Excelente trabajo con datos reales de gran escala!")
